# Genomic Data Preprocessing (Leakage-Free)

## Introduction

This notebook preprocesses the TCGA-BRCA RNA-seq gene expression data for breast cancer recurrence-free survival (RFS) prediction. The goal is to select the most variable genes and prepare genomic features for the multimodal Transformer model while preventing data leakage.

The notebook uses predefined training, validation, and test patient IDs. Gene filtering and selection are performed using **training data only**, and the same selected genes are used for all three datasets.

## Main Steps

1. **Load and inspect the data:** Load the raw RNA-seq dataset and check its dimensions, missing values, duplicate gene names, and patient IDs.
2. **Prepare patient IDs:** Standardize TCGA patient IDs and convert gene expression values to numeric format.
3. **Create dataset splits:** Match the genomic data with the predefined RFS training, validation, and test patient IDs.
4. **Remove zero-variance genes:** Calculate gene variance using training patients only and remove genes with zero variance.
5. **Remove near-zero-variance genes:** Apply a variance threshold of 0.01, calculated using the training set only.
6. **Rank genes by variance:** Rank the remaining genes according to their variance in the training set.
7. **Select the top 1,000 genes:** Choose the 1,000 highest-variance genes based only on training data.
8. **Align the datasets:** Ensure that training, validation, and test datasets contain exactly the same 1,000 genes in the same order.
9. **Prepare the final format:** Transpose the matrices so that each row represents a patient and each column represents a gene.
10. **Perform quality control:** Check missing values, infinite values, duplicate patient IDs, duplicate gene names, and feature statistics.
11. **Save the outputs:** Save the three processed genomic datasets and the selected gene list.

## Expected Output

The main leakage-free outputs are saved in the `genomic_preprocessing_leakage_free` folder:

- `genomic_train_1000_leakage_free.csv`
- `genomic_validation_1000_leakage_free.csv`
- `genomic_test_1000_leakage_free.csv`
- `selected_genes_1000_train_only.csv`

These files contain the same selected genes and are prepared for integration into the master RFS dataset.

## Conclusion

Gene filtering, variance ranking, and selection of the top 1,000 genes are based exclusively on the training set. The selected gene list is then applied consistently to the validation and test sets. This reduces the risk of data leakage and supports a more reliable evaluation of the model.

##Import Libraries & Configure Environment

In [1]:
# ============================================================
# GENOMIC PREPROCESSING PIPELINE
# Breast Cancer Recurrence Prediction
# Master's Thesis
# ============================================================

import os
import random
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("=" * 60)
print("Environment Ready")
print("=" * 60)

print(f"Numpy Version : {np.__version__}")
print(f"Pandas Version: {pd.__version__}")

Environment Ready
Numpy Version : 2.1.3
Pandas Version: 2.2.3


##Mount Google Drive & Define Paths

In [2]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================================
# Define Dataset Paths
# ============================================================

BASE_DIR = "/content/drive/MyDrive/BreastCanser-DataSet"

GENE_FILE = os.path.join(
    BASE_DIR,
    "Human__TCGA_BRCA__UNC__RNAseq__HiSeq_RNA__01_28_2016__BI__Gene__Firehose_RSEM_log2.tsv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "Genomic_Preprocessing"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Gene file:")
print(GENE_FILE)

print("\nOutput folder:")
print(OUTPUT_DIR)

Gene file:
/content/drive/MyDrive/BreastCanser-DataSet/Human__TCGA_BRCA__UNC__RNAseq__HiSeq_RNA__01_28_2016__BI__Gene__Firehose_RSEM_log2.tsv

Output folder:
/content/drive/MyDrive/BreastCanser-DataSet/Genomic_Preprocessing


##TEST cells:

In [4]:
import os
import numpy as np
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/TCGA_BRCA"
OUT_DIR = os.path.join(
    BASE_DIR, "genomic_preprocessing_leakage_free"
)
SPLIT_DIR = os.path.join(BASE_DIR, "dataset_split_rfs")

files = {
    "train": "genomic_train_1000_leakage_free.csv",
    "validation": "genomic_validation_1000_leakage_free.csv",
    "test": "genomic_test_1000_leakage_free.csv",
}

# Read saved matrices: first column contains patient IDs
data = {
    split: pd.read_csv(
        os.path.join(OUT_DIR, filename),
        index_col=0
    )
    for split, filename in files.items()
}

# Read the expected patient IDs
expected_ids = {
    "train": set(pd.read_csv(
        os.path.join(SPLIT_DIR, "train_rfs_patient_ids.csv")
    )["patient_id"].astype(str)),
    "validation": set(pd.read_csv(
        os.path.join(SPLIT_DIR, "validation_rfs_patient_ids.csv")
    )["patient_id"].astype(str)),
    "test": set(pd.read_csv(
        os.path.join(SPLIT_DIR, "test_rfs_patient_ids.csv")
    )["patient_id"].astype(str)),
}

print("=" * 70)
print("1. SHAPES AND PATIENT-ID CHECKS")
print("=" * 70)

for split, df in data.items():
    actual_ids = set(df.index.astype(str))

    print(f"\n{split.upper()}")
    print("Shape:", df.shape)
    print("Duplicate patient IDs:", df.index.duplicated().sum())
    print("Missing expected IDs:", len(expected_ids[split] - actual_ids))
    print("Unexpected IDs:", len(actual_ids - expected_ids[split]))
    print("Duplicate gene columns:", df.columns.duplicated().sum())

print("\n" + "=" * 70)
print("2. SPLIT OVERLAP")
print("=" * 70)

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    ids_a = set(data[a].index.astype(str))
    ids_b = set(data[b].index.astype(str))
    print(f"{a} vs {b} overlap:", len(ids_a & ids_b))

print("\n" + "=" * 70)
print("3. GENE ALIGNMENT")
print("=" * 70)

train_genes = data["train"].columns.tolist()

for split in ["validation", "test"]:
    genes = data[split].columns.tolist()
    print(
        f"Train vs {split}: same genes in same order?",
        genes == train_genes
    )

gene_list_path = os.path.join(
    OUT_DIR, "selected_genes_1000_train_only.csv"
)
selected_gene_list = pd.read_csv(gene_list_path)["gene"].astype(str).tolist()

print(
    "Saved gene list matches train columns?",
    selected_gene_list == train_genes
)
print("Number of selected genes:", len(train_genes))

print("\n" + "=" * 70)
print("4. NUMERIC VALUES AND MISSINGNESS")
print("=" * 70)

for split, df in data.items():
    numeric = df.apply(pd.to_numeric, errors="coerce")
    values = numeric.to_numpy(dtype=float)

    print(f"\n{split.upper()}")
    print("Missing values:", int(np.isnan(values).sum()))
    print("Infinite values:", int(np.isinf(values).sum()))
    print("Non-numeric values converted to missing:",
          int(numeric.isna().sum().sum() - df.isna().sum().sum()))
    print("All values finite:", bool(np.isfinite(values).all()))

1. SHAPES AND PATIENT-ID CHECKS

TRAIN
Shape: (249, 1000)
Duplicate patient IDs: 0
Missing expected IDs: 0
Unexpected IDs: 0
Duplicate gene columns: 0

VALIDATION
Shape: (54, 1000)
Duplicate patient IDs: 0
Missing expected IDs: 0
Unexpected IDs: 0
Duplicate gene columns: 0

TEST
Shape: (54, 1000)
Duplicate patient IDs: 0
Missing expected IDs: 0
Unexpected IDs: 0
Duplicate gene columns: 0

2. SPLIT OVERLAP
train vs validation overlap: 0
train vs test overlap: 0
validation vs test overlap: 0

3. GENE ALIGNMENT
Train vs validation: same genes in same order? True
Train vs test: same genes in same order? True
Saved gene list matches train columns? True
Number of selected genes: 1000

4. NUMERIC VALUES AND MISSINGNESS

TRAIN
Missing values: 0
Infinite values: 0
Non-numeric values converted to missing: 0
All values finite: True

VALIDATION
Missing values: 0
Infinite values: 0
Non-numeric values converted to missing: 0
All values finite: True

TEST
Missing values: 0
Infinite values: 0
Non-nume

##Load Dataset & Initial Inspection

In [ ]:
# ============================================================
# Load RNA-Seq Dataset
# ============================================================

gene_df = pd.read_csv(
    GENE_FILE,
    sep="\t"
)

print("=" * 60)
print("RAW DATASET")
print("=" * 60)

print(f"Shape: {gene_df.shape}")

print("\nFirst five rows:")
display(gene_df.head())

print("\nColumn names (first 10):")
print(gene_df.columns[:10].tolist())



# ============================================================
# Dataset Summary
# ============================================================

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

num_genes = gene_df.shape[0]
num_patients = gene_df.shape[1] - 1

print(f"Genes (rows)      : {num_genes:,}")
print(f"Patients (columns): {num_patients:,}")

print("\nGene column:")
print(gene_df.columns[0])

print("\nExample patient IDs:")
print(gene_df.columns[1:11].tolist())

print("\nMissing values in dataset:")
print(gene_df.isna().sum().sum())

print("\nDuplicate gene names:")
print(gene_df.iloc[:,0].duplicated().sum())

print("\nDuplicate patient columns:")
print(pd.Index(gene_df.columns[1:]).duplicated().sum())

RAW DATASET
Shape: (20155, 1094)

First five rows:


,attrib_name,TCGA.3C.AAAU,TCGA.3C.AALI,TCGA.3C.AALJ,TCGA.3C.AALK,TCGA.4H.AAAK,TCGA.5L.AAT0,TCGA.5L.AAT1,TCGA.5T.A9QA,TCGA.A1.A0SB,...,TCGA.UL.AAZ6,TCGA.UU.A93S,TCGA.V7.A7HQ,TCGA.W8.A86G,TCGA.WT.AB41,TCGA.WT.AB44,TCGA.XX.A899,TCGA.XX.A89A,TCGA.Z7.A8R5,TCGA.Z7.A8R6
0,A1BG,7.6300,7.8971,8.7287,7.5851,8.0762,7.6779,7.7299,8.3001,5.6496,...,5.4968,8.3353,10.0127,9.0608,6.9166,8.8835,7.8088,8.0008,8.7831,7.9619
1,A1CF,0.0000,0.0000,0.9310,0.0000,0.5115,0.0000,0.0000,0.6069,0.0000,...,0.4846,0.0000,0.0000,0.5603,0.0000,0.0000,0.0000,1.7492,0.0000,0.0000
2,A2BP1,0.0000,0.0000,0.0000,0.0000,2.2720,0.6659,0.0000,0.6069,2.4607,...,0.0000,0.0000,0.6922,2.6371,2.5190,0.0000,0.4789,1.2733,0.6756,0.0000
3,A2LD1,6.6999,6.1672,7.3422,5.9888,7.2796,6.8142,6.6861,7.8742,5.2816,...,7.5688,7.8208,6.9794,5.9240,8.2903,5.9711,7.0469,6.3386,6.3628,4.7108
4,A2ML1,1.2501,2.4196,0.0000,1.4087,2.1389,1.1198,2.3434,0.0000,2.0543,...,7.1771,0.0000,4.0885,1.2773,0.0000,2.9100,0.4789,5.8263,1.7610,2.3718



Column names (first 10):
['attrib_name', 'TCGA.3C.AAAU', 'TCGA.3C.AALI', 'TCGA.3C.AALJ', 'TCGA.3C.AALK', 'TCGA.4H.AAAK', 'TCGA.5L.AAT0', 'TCGA.5L.AAT1', 'TCGA.5T.A9QA', 'TCGA.A1.A0SB']
DATASET SUMMARY
Genes (rows)      : 20,155
Patients (columns): 1,093

Gene column:
attrib_name

Example patient IDs:
['TCGA.3C.AAAU', 'TCGA.3C.AALI', 'TCGA.3C.AALJ', 'TCGA.3C.AALK', 'TCGA.4H.AAAK', 'TCGA.5L.AAT0', 'TCGA.5L.AAT1', 'TCGA.5T.A9QA', 'TCGA.A1.A0SB', 'TCGA.A1.A0SD']

Missing values in dataset:
0

Duplicate gene names:
0

Duplicate patient columns:
0


##Convert Patient IDs

In [ ]:
# ============================================================
# Convert Patient IDs
# TCGA.A1.A0SB -> TCGA-A1-A0SB
# ============================================================

gene_df.columns = [
    gene_df.columns[0]
] + [
    c.replace(".", "-")
    for c in gene_df.columns[1:]
]

print("="*60)
print("Patient IDs Converted")
print("="*60)

print(gene_df.columns[:10])



Patient IDs Converted
Index(['attrib_name', 'TCGA-3C-AAAU', 'TCGA-3C-AALI', 'TCGA-3C-AALJ',
       'TCGA-3C-AALK', 'TCGA-4H-AAAK', 'TCGA-5L-AAT0', 'TCGA-5L-AAT1',
       'TCGA-5T-A9QA', 'TCGA-A1-A0SB'],
      dtype='object')


##Convert Gene Values to Numeric

In [ ]:
# ============================================================
# Convert Gene Values to Numeric
# Genomik değerleri sayısal forma çevirdim.
# bazı değerler metin formatında veya eksik olarak gelmiş olabilir.
# ============================================================

gene_df.iloc[:,1:] = gene_df.iloc[:,1:].apply(
    pd.to_numeric,
    errors="coerce"
)

print("="*60)
print("Numeric Conversion Finished")
print("="*60)

print(gene_df.dtypes.head())

Numeric Conversion Finished
attrib_name      object
TCGA-3C-AAAU    float64
TCGA-3C-AALI    float64
TCGA-3C-AALJ    float64
TCGA-3C-AALK    float64
dtype: object


##Add the RFS split IDs

In [ ]:
# ============================================================
# LOAD RFS TRAIN / VALIDATION / TEST PATIENT IDS
# ============================================================

import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/TCGA_BRCA"

SPLIT_DIR = os.path.join(
    BASE_DIR,
    "dataset_split_rfs"
)

train_ids = pd.read_csv(
    os.path.join(
        SPLIT_DIR,
        "train_rfs_patient_ids.csv"
    )
)["patient_id"].astype(str).tolist()

validation_ids = pd.read_csv(
    os.path.join(
        SPLIT_DIR,
        "validation_rfs_patient_ids.csv"
    )
)["patient_id"].astype(str).tolist()

test_ids = pd.read_csv(
    os.path.join(
        SPLIT_DIR,
        "test_rfs_patient_ids.csv"
    )
)["patient_id"].astype(str).tolist()


print("=" * 70)
print("RFS PATIENT SPLIT")
print("=" * 70)

print("Train patients      :", len(train_ids))
print("Validation patients :", len(validation_ids))
print("Test patients       :", len(test_ids))
print("Total RFS patients  :", (
    len(train_ids)
    + len(validation_ids)
    + len(test_ids)
))

RFS PATIENT SPLIT
Train patients      : 249
Validation patients : 54
Test patients       : 54
Total RFS patients  : 357


In [ ]:
# ============================================================
# CREATE RFS TRAIN / VALIDATION / TEST GENOMIC MATRICES
# ============================================================

print("=" * 70)
print("CREATING RFS GENOMIC MATRICES")
print("=" * 70)

# All genomic patient IDs from the raw RNA-seq dataset
genomic_patient_ids = gene_df.columns[1:].astype(str)

# Check which RFS patients exist in the genomic dataset
train_ids_found = [
    patient_id
    for patient_id in train_ids
    if patient_id in genomic_patient_ids
]

validation_ids_found = [
    patient_id
    for patient_id in validation_ids
    if patient_id in genomic_patient_ids
]

test_ids_found = [
    patient_id
    for patient_id in test_ids
    if patient_id in genomic_patient_ids
]


print("Train IDs found      :", len(train_ids_found))
print("Validation IDs found :", len(validation_ids_found))
print("Test IDs found       :", len(test_ids_found))


# Check that every RFS patient exists in the genomic dataset
assert len(train_ids_found) == len(train_ids), (
    "Some training patients are missing from the genomic dataset."
)

assert len(validation_ids_found) == len(validation_ids), (
    "Some validation patients are missing from the genomic dataset."
)

assert len(test_ids_found) == len(test_ids), (
    "Some test patients are missing from the genomic dataset."
)


# ------------------------------------------------------------
# Create train / validation / test genomic matrices
# ------------------------------------------------------------

gene_names = gene_df["attrib_name"].copy()


train_genomic = gene_df[
    ["attrib_name"] + train_ids_found
].copy()

validation_genomic = gene_df[
    ["attrib_name"] + validation_ids_found
].copy()

test_genomic = gene_df[
    ["attrib_name"] + test_ids_found
].copy()


print()
print("Train genomic matrix      :", train_genomic.shape)
print("Validation genomic matrix :", validation_genomic.shape)
print("Test genomic matrix       :", test_genomic.shape)

CREATING RFS GENOMIC MATRICES
Train IDs found      : 249
Validation IDs found : 54
Test IDs found       : 54

Train genomic matrix      : (20155, 250)
Validation genomic matrix : (20155, 55)
Test genomic matrix       : (20155, 55)


##Remove Zero Variance Genes

In [ ]:
# ============================================================
# REMOVE ZERO-VARIANCE GENES
# TRAIN SET ONLY
# ============================================================

print("=" * 70)
print("ZERO-VARIANCE FILTER")
print("=" * 70)

# Calculate gene variance using TRAIN patients only
train_gene_variance = train_genomic.iloc[:, 1:].var(axis=1)

# Keep genes with non-zero variance in the training set
non_zero_variance = train_gene_variance > 0

train_genomic = train_genomic.loc[
    non_zero_variance
].reset_index(drop=True)

validation_genomic = validation_genomic.loc[
    non_zero_variance
].reset_index(drop=True)

test_genomic = test_genomic.loc[
    non_zero_variance
].reset_index(drop=True)

# Keep the corresponding variance values
train_gene_variance = train_gene_variance.loc[
    non_zero_variance
].reset_index(drop=True)


print("Genes before filtering :", len(non_zero_variance))
print("Genes after filtering  :", len(train_genomic))

print()
print("Train shape      :", train_genomic.shape)
print("Validation shape :", validation_genomic.shape)
print("Test shape       :", test_genomic.shape)

ZERO-VARIANCE FILTER
Genes before filtering : 20155
Genes after filtering  : 20085

Train shape      : (20085, 250)
Validation shape : (20085, 55)
Test shape       : (20085, 55)


##Remove Near-Zero Variance Genes

In [ ]:
# ============================================================
# REMOVE NEAR-ZERO VARIANCE GENES
# TRAIN SET ONLY
# ============================================================

print("=" * 70)
print("NEAR-ZERO VARIANCE FILTER")
print("=" * 70)

# Calculate variance using TRAIN patients only
train_gene_variance = train_genomic.iloc[:, 1:].var(axis=1)

# Keep genes with variance greater than the threshold
threshold = 0.01

keep_genes = train_gene_variance > threshold

# Apply the same gene filter to all datasets
train_genomic = train_genomic.loc[
    keep_genes
].reset_index(drop=True)

validation_genomic = validation_genomic.loc[
    keep_genes
].reset_index(drop=True)

test_genomic = test_genomic.loc[
    keep_genes
].reset_index(drop=True)

# Keep corresponding training variances
train_gene_variance = train_gene_variance.loc[
    keep_genes
].reset_index(drop=True)


print("Variance threshold :", threshold)
print("Genes after filtering :", len(train_genomic))

print()
print("Train shape      :", train_genomic.shape)
print("Validation shape :", validation_genomic.shape)
print("Test shape       :", test_genomic.shape)

NEAR-ZERO VARIANCE FILTER
Variance threshold : 0.01
Genes after filtering : 19702

Train shape      : (19702, 250)
Validation shape : (19702, 55)
Test shape       : (19702, 55)


##Rank Genes by Variance

In [ ]:
# ============================================================
# RANK GENES BY VARIANCE
# TRAIN SET ONLY
# ============================================================

print("=" * 70)
print("RANKING GENES BY TRAINING VARIANCE")
print("=" * 70)

# Calculate variance using TRAIN patients only
train_gene_variance = train_genomic.iloc[:, 1:].var(axis=1)

# Store the variance temporarily
train_genomic["Variance"] = train_gene_variance.values

# Sort genes according to TRAIN variance
train_genomic = train_genomic.sort_values(
    "Variance",
    ascending=False
).reset_index(drop=True)

# Keep the same gene order for validation and test
gene_order = train_genomic["attrib_name"].tolist()

validation_genomic = (
    validation_genomic
    .set_index("attrib_name")
    .loc[gene_order]
    .reset_index()
)

test_genomic = (
    test_genomic
    .set_index("attrib_name")
    .loc[gene_order]
    .reset_index()
)

print("Remaining genes :", len(gene_order))

print()
print("Top 20 highest-variance genes:")
display(
    train_genomic[
        ["attrib_name", "Variance"]
    ].head(20)
)

RANKING GENES BY TRAINING VARIANCE
Remaining genes : 19702

Top 20 highest-variance genes:


,attrib_name,Variance
0,CLEC3A,27.549213
1,CPB1,27.403843
2,SCGB2A2,23.894340
3,TFF1,20.833184
4,SCGB1D2,19.600307
5,PIP,19.269050
6,GSTM1,18.653890
7,S100A7,18.416595
8,MUC6,17.795789
9,MUCL1,17.379038


##Select Top 1000 Genes

In [ ]:
# ============================================================
# SELECT TOP 1000 GENES
# BASED ON TRAINING VARIANCE ONLY
# ============================================================

print("=" * 70)
print("SELECTING TOP 1000 GENES")
print("=" * 70)

TOP_GENES = 1000

# Select the top 1000 genes according to
# variance calculated from TRAIN patients only
gene_top = train_genomic.head(
    TOP_GENES
).copy()

# Get the selected gene names
selected_genes_1000 = gene_top[
    "attrib_name"
].tolist()

print("Selected genes :", len(selected_genes_1000))

print()
print("First 20 selected genes:")

display(
    gene_top[
        ["attrib_name", "Variance"]
    ].head(20)
)

SELECTING TOP 1000 GENES
Selected genes : 1000

First 20 selected genes:


,attrib_name,Variance
0,CLEC3A,27.549213
1,CPB1,27.403843
2,SCGB2A2,23.894340
3,TFF1,20.833184
4,SCGB1D2,19.600307
5,PIP,19.269050
6,GSTM1,18.653890
7,S100A7,18.416595
8,MUC6,17.795789
9,MUCL1,17.379038


##Create the 1,000-gene train/validation/test matrices

In [ ]:
# ============================================================
# CREATE FINAL 1000-GENE GENOMIC MATRICES
# SAME GENES FOR TRAIN / VALIDATION / TEST
# ============================================================

print("=" * 70)
print("CREATING 1000-GENE GENOMIC MATRICES")
print("=" * 70)

# ------------------------------------------------------------
# Select the exact same 1000 genes in all datasets
# ------------------------------------------------------------

train_1000 = train_genomic[
    train_genomic["attrib_name"].isin(
        selected_genes_1000
    )
].copy()

validation_1000 = validation_genomic[
    validation_genomic["attrib_name"].isin(
        selected_genes_1000
    )
].copy()

test_1000 = test_genomic[
    test_genomic["attrib_name"].isin(
        selected_genes_1000
    )
].copy()


# ------------------------------------------------------------
# Put genes in exactly the same order
# as the training selection
# ------------------------------------------------------------

train_1000 = (
    train_1000
    .set_index("attrib_name")
    .loc[selected_genes_1000]
    .reset_index()
)

validation_1000 = (
    validation_1000
    .set_index("attrib_name")
    .loc[selected_genes_1000]
    .reset_index()
)

test_1000 = (
    test_1000
    .set_index("attrib_name")
    .loc[selected_genes_1000]
    .reset_index()
)


print("Train genes      :", len(train_1000))
print("Validation genes :", len(validation_1000))
print("Test genes       :", len(test_1000))

print()
print("Train shape      :", train_1000.shape)
print("Validation shape :", validation_1000.shape)
print("Test shape       :", test_1000.shape)

CREATING 1000-GENE GENOMIC MATRICES
Train genes      : 1000
Validation genes : 1000
Test genes       : 1000

Train shape      : (1000, 251)
Validation shape : (1000, 55)
Test shape       : (1000, 55)


##Transpose to patient-level format

In [ ]:
# ============================================================
# REMOVE TEMPORARY VARIANCE COLUMN
# AND CREATE PATIENT-LEVEL GENOMIC MATRICES
# ============================================================

print("=" * 70)
print("CREATING PATIENT-LEVEL 1000-GENE MATRICES")
print("=" * 70)

# Remove temporary Variance column
if "Variance" in train_1000.columns:
    train_1000 = train_1000.drop(
        columns=["Variance"]
    )

# ------------------------------------------------------------
# Genes -> columns
# Patients -> rows
# ------------------------------------------------------------

train_patient = (
    train_1000
    .set_index("attrib_name")
    .T
)

validation_patient = (
    validation_1000
    .set_index("attrib_name")
    .T
)

test_patient = (
    test_1000
    .set_index("attrib_name")
    .T
)

# ------------------------------------------------------------
# Set patient ID as index
# ------------------------------------------------------------

train_patient.index.name = "patient_id"
validation_patient.index.name = "patient_id"
test_patient.index.name = "patient_id"


print("Train shape      :", train_patient.shape)
print("Validation shape :", validation_patient.shape)
print("Test shape       :", test_patient.shape)

print()
print("Train patients      :", train_patient.shape[0])
print("Validation patients :", validation_patient.shape[0])
print("Test patients       :", test_patient.shape[0])

print()
print("Train genes      :", train_patient.shape[1])
print("Validation genes :", validation_patient.shape[1])
print("Test genes       :", test_patient.shape[1])

CREATING PATIENT-LEVEL 1000-GENE MATRICES
Train shape      : (249, 1000)
Validation shape : (54, 1000)
Test shape       : (54, 1000)

Train patients      : 249
Validation patients : 54
Test patients       : 54

Train genes      : 1000
Validation genes : 1000
Test genes       : 1000


##SAVE LEAKAGE-FREE 1000-GENE GENOMIC DATASETS

In [ ]:
# ============================================================
# SAVE LEAKAGE-FREE 1000-GENE GENOMIC DATASETS
# ============================================================

GENOMIC_OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "genomic_preprocessing_leakage_free"
)

os.makedirs(
    GENOMIC_OUTPUT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Save train / validation / test
# ------------------------------------------------------------

train_patient.to_csv(
    os.path.join(
        GENOMIC_OUTPUT_DIR,
        "genomic_train_1000_leakage_free.csv"
    )
)

validation_patient.to_csv(
    os.path.join(
        GENOMIC_OUTPUT_DIR,
        "genomic_validation_1000_leakage_free.csv"
    )
)

test_patient.to_csv(
    os.path.join(
        GENOMIC_OUTPUT_DIR,
        "genomic_test_1000_leakage_free.csv"
    )
)


# ------------------------------------------------------------
# Save the selected 1000 gene list
# ------------------------------------------------------------

pd.DataFrame({
    "gene": selected_genes_1000
}).to_csv(
    os.path.join(
        GENOMIC_OUTPUT_DIR,
        "selected_genes_1000_train_only.csv"
    ),
    index=False
)


print("=" * 70)
print("LEAKAGE-FREE GENOMIC DATASETS SAVED")
print("=" * 70)

print()
print("Output directory:")
print(GENOMIC_OUTPUT_DIR)

print()
print("Files:")

for filename in sorted(
    os.listdir(GENOMIC_OUTPUT_DIR)
):
    print(" -", filename)

LEAKAGE-FREE GENOMIC DATASETS SAVED

Output directory:
/content/drive/MyDrive/TCGA_BRCA/genomic_preprocessing_leakage_free

Files:
 - genomic_test_1000_leakage_free.csv
 - genomic_train_1000_leakage_free.csv
 - genomic_validation_1000_leakage_free.csv
 - selected_genes_1000_train_only.csv


##Save Selected Gene List

In [ ]:
# ============================================================
# Save Gene List
# ============================================================

gene_top[["attrib_name","Variance"]].to_csv(

    os.path.join(
        OUTPUT_DIR,
        "selected_gene_list.csv"
    ),

    index=False
)

print("Gene list saved.")

Gene list saved.


##Create Patient-Level Genomic Matrix (Transpose)

In [ ]:
# ============================================================
# Create Patient-Level Genomic Matrix
# Genes -> Columns
# Patients -> Rows
# Veri formatını değiştirdim. Her satır hasta, her sütun gen bilgisini temsil ediyor.
# ============================================================


import pandas as pd
import os


print("="*70)
print("CREATING PATIENT-LEVEL GENOMIC MATRIX")
print("="*70)


# Remove variance column if exists
if "Variance" in gene_top.columns:
    gene_matrix = gene_top.drop(
        columns=["Variance"]
    )
else:
    gene_matrix = gene_top.copy()


# Set gene names as index

gene_matrix = gene_matrix.set_index(
    "attrib_name"
)


# Transpose

genomic_patient = gene_matrix.T


# Rename index

genomic_patient.index.name = "patient_id"


print("Patients :", genomic_patient.shape[0])
print("Genes    :", genomic_patient.shape[1])


print("\nFirst rows:")
display(genomic_patient.head())

CREATING PATIENT-LEVEL GENOMIC MATRIX
Patients : 1093
Genes    : 1000

First rows:


attrib_name,CLEC3A,CPB1,SCGB2A2,SCGB1D2,TFF1,GSTM1,PIP,S100A7,MUCL1,CYP2B7P1,...,DNAH5,DMKN,SLC1A6,PTGDS,TFCP2L1,ZIC5,CRYAB,EMILIN3,TNNT2,INA
patient_id,,,,,,,,,,,,,,,,,,,,,
TCGA-3C-AAAU,2.4543,16.1613,15.9692,12.2314,6.3268,10.8344,13.0549,0.000,8.3070,9.7704,...,8.1773,6.6771,0.4273,6.7059,9.5730,0.4273,6.4739,2.8300,5.6322,1.4454
TCGA-3C-AALI,11.7687,4.2437,16.1618,10.5341,7.5467,6.5291,8.3734,0.000,7.3052,2.6865,...,7.1481,3.9710,3.1065,8.3567,3.8673,5.1399,7.2288,3.1974,6.7710,1.0618
TCGA-3C-AALJ,0.0000,11.1077,0.0000,0.9310,7.8249,2.8770,0.0000,0.000,11.3664,9.9113,...,6.0306,0.0000,0.0000,6.8255,5.5054,4.1143,9.1020,4.1278,6.3032,1.4922
TCGA-3C-AALK,5.3766,6.9285,8.7442,6.5496,13.5956,11.6371,11.5943,6.214,12.0994,9.1587,...,7.1961,7.8827,0.0000,8.5023,6.6181,0.0000,8.7161,3.8877,3.6054,0.4995
TCGA-4H-AAAK,7.4930,2.7075,14.9834,9.3356,12.3546,11.3659,7.6139,0.000,7.2726,9.6403,...,8.6912,8.6382,0.0000,7.4410,9.2873,0.0000,8.7379,4.3130,1.4341,1.4341


##StandardScaler Normalization

In [ ]:
# ============================================================
# Standardization
# Gen değerlerini aynı ölçeğe getirdim.
# Ortalama değeri sıfıra, standart sapmayı bire yaklaştırıyoruz. Böylece model genleri daha dengeli öğrenebilir.
# ============================================================

from sklearn.preprocessing import StandardScaler


print("="*70)
print("STANDARDIZATION")
print("="*70)


scaler = StandardScaler()


scaled_values = scaler.fit_transform(
    genomic_patient
)


genomic_scaled = pd.DataFrame(
    scaled_values,
    index=genomic_patient.index,
    columns=genomic_patient.columns
)


print("Shape:")
print(genomic_scaled.shape)


print("\nMean:")
print(genomic_scaled.mean().mean())


print("\nStd:")
print(genomic_scaled.std().mean())

STANDARDIZATION
Shape:
(1093, 1000)

Mean:
-4.551399117614226e-17

Std:
1.0004577706808773


##Quality Control

In [ ]:
# ============================================================
# QUALITY CONTROL
# kalite kontrol yaptım
# ============================================================


print("="*70)
print("GENOMIC QUALITY CONTROL")
print("="*70)


qc = {}


qc["patients"] = genomic_scaled.shape[0]

qc["genes"] = genomic_scaled.shape[1]


qc["missing_values"] = (
    genomic_scaled.isna()
    .sum()
    .sum()
)


qc["infinite_values"] = (
    (~genomic_scaled.replace(
        [float("inf"), float("-inf")],
        pd.NA
    ).notna())
    .sum()
    .sum()
)


qc["duplicate_patients"] = (
    genomic_scaled.index.duplicated()
    .sum()
)


qc["duplicate_genes"] = (
    genomic_scaled.columns.duplicated()
    .sum()
)



for k,v in qc.items():
    print(f"{k}: {v}")


print("\nValue distribution:")
display(
    genomic_scaled.describe()
)

GENOMIC QUALITY CONTROL
patients: 1093
genes: 1000
missing_values: 0
infinite_values: 0
duplicate_patients: 0
duplicate_genes: 0

Value distribution:


attrib_name,CLEC3A,CPB1,SCGB2A2,SCGB1D2,TFF1,GSTM1,PIP,S100A7,MUCL1,CYP2B7P1,...,DNAH5,DMKN,SLC1A6,PTGDS,TFCP2L1,ZIC5,CRYAB,EMILIN3,TNNT2,INA
count,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,...,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03
mean,-6.338327e-16,1.461066e-15,1.015758e-15,1.733289e-15,4.098785e-15,6.760882e-16,2.642595e-15,4.810628e-16,1.950255e-15,-2.678350e-15,...,-1.602459e-15,-6.105922e-15,-3.607971e-16,-8.727389e-16,3.685981e-15,1.495195e-16,-1.898248e-15,-1.059638e-15,-9.426230e-16,9.044305e-16
std,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,...,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00
min,-9.178925e-01,-1.267645e+00,-1.891635e+00,-1.636181e+00,-1.916251e+00,-1.298324e+00,-1.973497e+00,-8.530970e-01,-1.712813e+00,-1.869744e+00,...,-3.600830e+00,-3.726541e+00,-5.808467e-01,-3.590751e+00,-3.428517e+00,-5.997344e-01,-2.315402e+00,-1.698349e+00,-1.043650e+00,-8.790223e-01
25%,-9.178925e-01,-8.394311e-01,-7.472521e-01,-8.098245e-01,-6.670097e-01,-9.657040e-01,-7.132956e-01,-8.530970e-01,-7.295261e-01,-9.426050e-01,...,-4.901390e-01,-4.639973e-01,-5.808467e-01,-6.104348e-01,-7.046167e-01,-5.997344e-01,-7.199982e-01,-7.015546e-01,-8.279823e-01,-6.283451e-01
50%,-4.611182e-01,-1.446340e-01,4.694128e-02,-1.902028e-02,2.647548e-01,-1.411223e-02,1.179782e-01,-4.560674e-01,-9.968968e-02,1.745996e-01,...,1.941841e-01,1.642518e-01,-3.974506e-01,8.355133e-03,2.435746e-02,-5.997344e-01,-1.288497e-01,-1.676008e-01,-2.722322e-01,-2.959168e-01
75%,8.124130e-01,6.025623e-01,8.439793e-01,7.429316e-01,7.731723e-01,1.013761e+00,7.587249e-01,6.213906e-01,6.712078e-01,8.145406e-01,...,7.063030e-01,5.682263e-01,6.498216e-02,6.938830e-01,6.746361e-01,1.036434e-01,5.695206e-01,4.917330e-01,6.014767e-01,1.743451e-01
max,2.508348e+00,2.789720e+00,2.281520e+00,2.825615e+00,1.991044e+00,2.260445e+00,2.774407e+00,3.199846e+00,2.840769e+00,2.053867e+00,...,2.225314e+00,3.944044e+00,5.232145e+00,4.057156e+00,2.881160e+00,3.633310e+00,3.440260e+00,4.087329e+00,4.762728e+00,4.920262e+00


##Save Final Genomic Dataset

In [ ]:
# ============================================================
# SAVE FINAL OUTPUTS
# preprocessing sonrası klinik veri dosyasını kaydettim
# ============================================================


OUTPUT_DIR = "/content/drive/MyDrive/TCGA_BRCA/genomic_processed"


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# Save processed genomic matrix

genomic_scaled.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "genomic_processed.csv"
    )
)



# Save statistics

gene_statistics = pd.DataFrame({

    "gene":
    genomic_scaled.columns,

    "mean":
    genomic_scaled.mean(),

    "std":
    genomic_scaled.std(),

    "variance":
    genomic_scaled.var()

})


gene_statistics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "gene_statistics.csv"
    ),
    index=False
)



# Save preprocessing report

with open(
    os.path.join(
        OUTPUT_DIR,
        "preprocessing_report.txt"
    ),
    "w"
) as f:

    f.write(
f"""
TCGA-BRCA RNA-seq preprocessing report

Input:
Firehose RNA-seq log2 normalized expression

Final dataset:

Patients:
{genomic_scaled.shape[0]}

Genes:
{genomic_scaled.shape[1]}


Preprocessing:

- TCGA patient IDs converted
- Duplicate genes checked
- Duplicate patients checked
- Missing values checked
- Zero variance genes removed
- Near-zero variance genes removed
- Top 1000 variable genes selected
- Data transposed to patient-level format
- StandardScaler normalization applied


Quality control:

Missing values:
{qc["missing_values"]}

Infinite values:
{qc["infinite_values"]}

Duplicate patients:
{qc["duplicate_patients"]}

Duplicate genes:
{qc["duplicate_genes"]}

"""
    )


print("="*70)
print("SAVED SUCCESSFULLY")
print("="*70)


print(
os.listdir(OUTPUT_DIR)
)

SAVED SUCCESSFULLY
['genomic_processed.csv', 'gene_statistics.csv', 'preprocessing_report.txt']
